In [1]:
import google.generativeai as genai
import pandas as pd
from datasets import load_dataset
import os
from dotenv import load_dotenv


d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(model_name='gemma-3-1b-it')

In [5]:
def generate_gemma_response(prompt):
    response = model.generate_content(prompt,
                                          generation_config={
        'temperature': 0.7,
        'top_p': 0.95,
        'top_k': 40,
        'max_output_tokens': 8046,
        'stop_sequences': ['</answer>'],  # useful for your format
    }
    )
    return response.text


def gemma_evolution(base_question:str,raw_reasoning:str=None, refined_reasoning:str=None):

    normal_response = generate_gemma_response(base_question)
    print("Normal Response:\n", normal_response)
    print("-----------------------------------------------")

    if raw_reasoning is not None:
        raw_reasoned_response = generate_gemma_response(base_question+raw_reasoning)
        print("\nRaw Reasoned Response:\n", raw_reasoned_response)
        print("-----------------------------------------------")
    else:
        raw_reasoned_response = None
    if refined_reasoning is not None:
        refined_reasoned_response = generate_gemma_response(base_question+refined_reasoning)
        print("\nRefined Reasoned Response:\n", refined_reasoned_response)
    else:
        refined_reasoned_response = None
    
    
    return normal_response, raw_reasoned_response, refined_reasoned_response


import re

def parse_scores(response_text):
    """
    Parse score response from Gemini into structured format
    """
    scores = {
        'clarity': 0,
        'completeness': 0,
        'correctness': 0,
        'structure': 0,
        'overall': 0
    }
    
    # Pattern to match "Criteria: X/10" format
    patterns = {
        'clarity': r'Clarity:?\s*(\d+)(?:/10)?',
        'completeness': r'Completeness:?\s*(\d+)(?:/10)?',
        'correctness': r'Correctness:?\s*(\d+)(?:/10)?',
        'structure': r'Structure:?\s*(\d+)(?:/10)?',
        'overall': r'Overall:?\s*(\d+)(?:/10)?'
    }
    
    for criterion, pattern in patterns.items():
        match = re.search(pattern, response_text, re.IGNORECASE)
        if match:
            scores[criterion] = int(match.group(1))
    
    return scores

def score_reasoning_quality(question, reasoning):
    prompt = f"""Evaluate this reasoning on a scale of 0-10 for the following criteria:

Question: {question}

Reasoning: {reasoning}


Score the following:
1. Clarity (0-10): Is the reasoning easy to follow?
2. Completeness (0-10): Are all steps explained?
3. Correctness (0-10): Is the logic sound?
4. Structure (0-10): Is it well-organized?

Provide scores in this format:
Clarity: X/10
Completeness: X/10
Correctness: X/10
Structure: X/10
Overall: X/10"""

    response = model.generate_content(prompt)
    return parse_scores(response.text)



In [29]:
print(generate_gemma_response("Compute \[\sum_{n=1}^{1000} \ frac{1}{n^2 + n}.\]"))

<>:1: SyntaxWarning: invalid escape sequence '\['
<>:1: SyntaxWarning: invalid escape sequence '\['
C:\Users\hazar\AppData\Local\Temp\ipykernel_3424\256456054.py:1: SyntaxWarning: invalid escape sequence '\['
  print(generate_gemma_response("Compute \[\sum_{n=1}^{1000} \ frac{1}{n^2 + n}.\]"))


Let $S = \sum_{n=1}^{1000} \frac{1}{n^2 + n}$. We can write
\[ \frac{1}{n^2 + n} = \frac{1}{n(n+1)} = \frac{1}{n} - \frac{1}{n+1}. \]
Then
\[ S = \sum_{n=1}^{1000} \left( \frac{1}{n} - \frac{1}{n+1} \right) = \left( \frac{1}{1} - \frac{1}{2} \right) + \left( \frac{1}{2} - \frac{1}{3} \right) + \cdots + \left( \frac{1}{1000} - \frac{1}{1001} \right). \]
This is a telescoping sum, so
\[ S = 1 - \frac{1}{1001} = \frac{1001 - 1}{1001} = \frac{1000}{1001}. \]
Therefore,
\[ \sum_{n=1}^{1000} \frac{1}{n^2 + n} = \frac{1000}{1001}. \]

Final Answer: The final answer is $\boxed{1000/1001}$


1. Math Question

In [ ]:
##
normal_response, raw_reasoned_response, refined_reasoned_response=gemma_evolution(
    base_question="From a group of six students, how many different four-student committees can be chosen?",
    raw_reasoning=f"""<reasoning>We need number of ways to choose 4 students from 6: C(6,4)=30. Provide answer.</reasoning>""",
    refined_reasoning=f"""<reasoning>
This is a combination problem: choosing 4 students from 6 where order doesn't matter.

Let me calculate C(6,4):
- Formula: C(n,r) = n! / (r!(n-r)!)
- Substituting: C(6,4) = 6! / (4!·2!)

Let me break down the calculation

Let me validate the calculation

Let me verify the answer to question asked
</reasoning>"""
                      )

Normal Response:
 Let $n$ be the number of students in the group, so $n=6$.
We want to choose a four-student committee from the group of six students.
The order in which the students are chosen does not matter, so we are looking for the number of combinations of choosing 4 students from a group of 6 students.
The number of combinations of choosing $k$ items from a set of $n$ items is given by the binomial coefficient $\binom{n}{k} = \frac{n!}{k!(n-k)!}$, where $n!$ is the factorial of $n$.
In this case, we want to choose a four-student committee from a group of six students, so we have $n=6$ and $k=4$.
The number of different four-student committees is $\binom{6}{4} = \frac{6!}{4!(6-4)!} = \frac{6!}{4!2!} = \frac{6 \times 5 \times 4 \times 3 \times 2 \times 1}{(4 \times 3 \times 2 \times 1)(2 \times 1)} = \frac{6 \times 5}{2 \times 1} = \frac{30}{2} = 15$.

Thus, there are 15 different four-student committees that can be chosen from a group of six students.

Final Answer: The final ans

In [ ]:
print(generate_gemma_response(f"""This is a question you get: From a group of six students, how many different four-student committees can be chosen?
                              
                              You got 2 reasoning to answer this question:

1. [Problem Understanding]
We need to find how many different 4-student committees can be formed from 6 students. Since the order doesn't matter (a committee is the same regardless of selection order), this is a combinations problem.

[Solution]
Using the combinations formula C(n,k) = n!/(k!(n-k)!):

C(6,4) = 6!/(4!×2!)
       = 720/(24×2)
       = 720/48
       = 15
                              
2. The user asks: From a group of six students, how many different four-student committees can be chosen? That's a simple combinatorial problem: number of ways to choose 4 out of 6 = C(6,4) = 15. Provide answer.
Potentially explain calculation: C(n,k) = n!/(k!(n-k)!). So C(6,4) = 6!/(4!2!) = (720)/(24*2)=720/48=15. Or note symmetry C(6,4)=C(6,2)=15.
Answer: 15 committees.
Thus final.



       
       
Which option is the best for you"""))

Option 2 is the best and most concise explanation. It directly addresses the problem and provides the correct answer.

Here’s a breakdown of why Option 2 is superior:

* **Directness:** It immediately states the solution – C(6,4) = 15.
* **Explanation:** It clearly explains the formula and how to calculate it, making it easier to understand.
* **Conciseness:** It avoids unnecessary elaboration.

Option 1, while correct, is a bit more verbose and doesn't fully explain the underlying concept.

Therefore, **Option 2 is the best.**



In [ ]:
question=f"From a group of six students, how many different four-student committees can be chosen?"
option1_reasoning=f"""[Problem Understanding]
We need to find how many different 4-student committees can be formed from 6 students. Since the order doesn't matter (a committee is the same regardless of selection order), this is a combinations problem.

[Solution]
Using the combinations formula C(n,k) = n!/(k!(n-k)!):

C(6,4) = 6!/(4!×2!)
       = 720/(24×2)
       = 720/48
       = 15"""
option2_reasoning=f"""The user asks: From a group of six students, how many different four-student committees can be chosen? That's a simple combinatorial problem: number of ways to choose 4 out of 6 = C(6,4) = 15. Provide answer.
Potentially explain calculation: C(n,k) = n!/(k!(n-k)!). So C(6,4) = 6!/(4!2!) = (720)/(24*2)=720/48=15. Or note symmetry C(6,4)=C(6,2)=15."""
option1_scores = score_reasoning_quality(question, option1_reasoning)
option2_scores = score_reasoning_quality(question, option2_reasoning)

print("Option 1 (Structured):", option1_scores)
print("Option 2 (Unstructured):", option2_scores)

Option 1 (Structured): {'clarity': 8, 'completeness': 9, 'correctness': 10, 'structure': 9, 'overall': 9}
Option 2 (Unstructured): {'clarity': 9, 'completeness': 10, 'correctness': 10, 'structure': 10, 'overall': 9}


In [ ]:
question=f"""Aunt Angela has 70 jellybeans in a jar.  She wants to divide them equally and give them to her 3 nephews and x nieces.  How many jellybeans did each nephew or niece receive?
If we know the answer to the above question is 14, what is the value of unknown variable x?"""
option1_reasoning=f"""[Problem Understanding]
We need to find the value of x (number of nieces) given that:
- Aunt Angela has 70 jellybeans total
- She divides them equally among 3 nephews and x nieces
- Each person receives 14 jellybeans

[Approach Selection]
Since the jellybeans are divided equally, we can use the equation:
Total jellybeans = Number of people × Jellybeans per person

The total number of people is (3 + x), so:
70 = (3 + x) × 14

[Step-by-Step Solution]
Starting with: 70 = (3 + x) × 14

Divide both sides by 14:
70 ÷ 14 = 3 + x
5 = 3 + x

Subtract 3 from both sides:
5 - 3 = x
x = 2

[Verification]
Let me check if x = 2 gives the correct answer:
- Total people: 3 nephews + 2 nieces = 5 people
- Jellybeans per person: 70 ÷ 5 = 14 ✓
- This matches the given information

[Confidence]
High confidence. The algebra is straightforward and the verification confirms our answer."""
option2_reasoning=f"""We need to parse the problem.
Aunt Angela has 70 jellybeans in a jar. She wants to divide them equally and give them to her 3 nephews and x nieces. So total recipients = 3 nephews + x nieces = 3 + x people. She divides 70 equally among them, each gets same number. The answer to "How many jellybeans did each nephew or niece receive?" is 14. So each gets 14 jellybeans. So 70 divided by (3 + x) = 14. Solve: 70 = 14 (3 + x). So 70 = 42 + 14x. Then 14x = 28, so x = 2. So answer: x = 2.
Thus there are 2 nieces. Also we can mention that the total recipients = 5, each gets 14.
Thus answer: x = 2.
Now produce final answer."""
option1_scores = score_reasoning_quality(question, option1_reasoning)
option2_scores = score_reasoning_quality(question, option2_reasoning)

print("Option 1 (Structured):", option1_scores)
print("Option 2 (Unstructured):", option2_scores)

Option 1 (Structured): {'clarity': 8, 'completeness': 9, 'correctness': 9, 'structure': 9, 'overall': 0}
Option 2 (Unstructured): {'clarity': 8, 'completeness': 10, 'correctness': 9, 'structure': 9, 'overall': 9}


In [7]:
print(generate_gemma_response(f"""Context:
Formula, Vol. 1: Formula, Vol.  1 is the debut studio album by American singer Romeo Santos, released on November 8, 2011 by Sony Music Latin.  It is Santos's first album as a solo artist following the break-up of American bachata group Aventura, of which he was the lead singer.  The record contains fifteen tracks, most of which were composed by Santos and co-produced with Ivan Chevere.  The album experiments with the sound of bachata and other genres including R&B and flamenco.  It features several Anglophone and Hispanophone guest artists including Usher, Tomatito, Mario Domm, and Lil Wayne.  Recording for the album took place in 2011 at The Castle, Fight Klub, and EMG Studios in New York City.  A deluxe edition of the album containing five extra tracks was released exclusively in Walmart retail stores in the United States.

Cry Baby (Melanie Martinez album): Cry Baby is the debut studio album by American recording artist Melanie Martinez.  It was released on August 14, 2015, by Atlantic Records through digital download, CD and vinyl.  The album was supported by the release three singles.  Two singles preceded the album's release: lead single "Pity Party", was released on June 2, 2015, while the second single, "Soap", was released on July 10, 2015.  The third and final single from the album, "Sippy Cup" was released July 31, 2015.  "Cry Baby" is a visual concept album.  The album is labeled as an alternative pop, electropop, and indie pop release, and received generally positive reviews from critics.  In February 2017, the album was certified Platinum, after earning 1 million album-equivalent units (sales, streaming and track sales).

Lift Off (song): "Lift Off" is a song by Jay-Z and Kanye West featuring American recording artist Beyoncé.  It was written by Kanye West, Jay-Z, Jeff Bhasker, Mike Dean, Bruno Mars and Seal, while production was handled by West, Bhasker, Mike Dean, Pharrell, Q-Tip, and Don Jazzy for Jay-Z' s and West' s collaboration album, "Watch the Throne" (2011).  The song was rumored to be released as the lead single from the album containing additional vocals by Bruno Mars.  However, Mars never appeared on the song and it was sent to urban contemporary radio on August 23, 2011.

Thursday / Envy: Thursday / Envy is a split album containing tracks contributed by the screamo acts Thursday and Envy.  It was released exclusively in a package containing the album on both 180 gram 12" vinyl and on CD — individual CDs or vinyl have not been made available.  Three limited screen printed editions have been made available exclusively through web stores as of September 15, 2008.  The album has since seen a limited release in cassette format, all 500 copies of which were sold exclusively through independent record label, Academy Fight Song's web store.

List of songs recorded by Melanie Martinez: Melanie Adele Martinez is an American singer and songwriter.  Melanie Martinez auditioned for the American television vocal talent show" The Voice" and became a member of Team Adam.  Within the fifth week, she was eliminated which subsequently led to her beginning independent work on original material.  In 2014, she released "Dollhouse", her debut EP featuring the singles "Dollhouse" and "Carousel", which appeared on the trailer for the television series "".  In 2015, Martinez released her conceptual debut album titled "Cry Baby", featuring the critically acclaimed lead single "Pity Party", along with other singles "Soap" and "Sippy Cup".

Sippy Cup (song): "Sippy Cup" is a song by American recording artist Melanie Martinez as the third single from her debut album, "Cry Baby".  A music video featuring fashion designer Stella Rose Saint Clair was released July 31, 2015.

Swedish House Mafia discography: Swedish house music supergroup Swedish House Mafia has released two compilation albums, one live album, six singles and six music videos.  The trio formed during the mid-2000s whilst touring together and assisting each other in music production, and recorded and released the single "Leave the World Behind" in 2009, although all were assigned individual artist credits for the song.  The group itself rose to prominence after performing a DJ set at the Cream Amnesia nightclub in Ibiza in 2008, and were later signed by record label EMI in 2010. " Until One", a compilation album containing both their own material and mixes of songs by other artists, peaked at number 13 in Sweden and in the top ten of the album charts of The Netherlands and the Flanders region of Belgium: two singles, "One" and "Miami 2 Ibiza", both attained chart success across Europe, with the former topping the Dutch Top 40 and the latter reaching the top five of the Flanders, Irish and United Kingdom singles charts.  "One" and "Miami 2 Ibiza" were certified septuple and quintuple platinum in Sweden by the International Federation of the Phonographic Industry" (IFPI).

Benji Hughes: Benji Hughes is an American musical artist from Charlotte, North Carolina.  On July 22, 2008, Benji Hughes released his debut album, entitled "A Love Extreme" on New West Records.  "A Love Extreme" is a double-disc album containing 25 songs.  It was recorded with acclaimed producer and Los Angeles session musician Keefus Ciancia.  Hughes' live band during this period contained a rotating cast of notable members, including Barbara Gruska (of Jenny Lewis and The Belle Brigade) on drums; two of Hughes' former Muscadine bandmates: solo-artist and producer Jonathan Wilson on guitar and backing vocals, and Stacy Leazer on bass; solo artist and producer Jon Lindsay on keyboards and backing vocals; Ciancia on keyboards; and veteran Charlotte musicians Peter Gray (guitar) and David Kim (drums), among others.  The album received favorable reviews from some prominent critics, including Jon Pareles of "The New York Times" and Chuck Klosterman of "Esquire", but it sold few copies.

The Damage (Marillion song): "The Damage" is a song by British neo-progressive rock band Marillion which appeared on their 13th studio album, "Marbles", released in May 2004.  In October 2005, a one-disc live album containing a subset of the full two-disc studio version entitled "Marbles Live" was released to retail shops in the UK.  The recording was made at the London Astoria in July 2004.  To promote this album, the track "The Damage" was made available as a digital download; it is thus the third song to be released from "Marbles" and the only track to be released from "Marbles Live".  Download-only releases were not yet eligible to chart on the UK Singles Chart at the time, but the single did reach #2 on the UK Official Download Chart.  There was no physical release available, but a one-track CD version was sent out as a promo (source of cover art).

Melanie Martinez (singer): Melanie Adele Martinez ( ; born April 28, 1995) is an American singer, songwriter, music video director, and photographer.  Born in Astoria, Queens and raised in Baldwin, New York on Long Island, she first participated in the "MSG Varsity Talent Show" during her junior year of high school, and subsequently rose to prominence in 2012 after appearing on the American television vocal talent show "The Voice."  She auditioned singing Britney Spears's "Toxic", and made it to the Top 6 before being eliminated in the fifth week of live shows.

Question: From where is the artist who made the album containing the single "Sippy Cup"?"""))

The artist who made the album containing the single "Sippy Cup" is Melanie Martinez.
